In [9]:
import os
import sys
import subprocess

# Install required packages in the current notebook environment if missing
def install_if_missing(package):
    try:
        __import__(package.replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

install_if_missing("openai")
install_if_missing("python-dotenv")

from openai import OpenAI
from dotenv import load_dotenv

# Optional: loads .env only if it exists locally
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY is not set. Please set it as an environment variable before running this notebook."
    )

client = OpenAI(api_key=api_key)

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 3.4 MB/s  0:00:0036m-:--:--
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 7.8 MB/s  0:00:00 eta 0:00:01
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (1


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [11]:
# ============================================================
# Section 1: Generative AI Model Selection & Setup
# ============================================================

"""
Model Selected: GPT-4o-mini by OpenAI

Rationale:
- Industry-standard LLM with excellent instruction-following ability
- Fast response times and cost-effective for testing multiple templates
- Strong performance on structured, domain-specific advice generation
- Well-documented API that supports role-based prompting (system/user)
- Widely used in production advice and recommendation systems
"""

import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def call_gpt(prompt: str, system_message: str = None, max_tokens: int = 500) -> str:
    """
    Helper function to call GPT and return the response text.

    Args:
        prompt: The user's message/prompt
        system_message: Optional system-level instruction to set the model's role
        max_tokens: Maximum response length

    Returns:
        str: The model's text response
    """
    messages = []

    if system_message:
        messages.append({"role": "system", "content": system_message})

    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        max_tokens=max_tokens,
        temperature=0.7  # slight creativity while staying factual
    )

    return response.choices[0].message.content

# ---- Test Connection ----
test = call_gpt("Confirm you are connected and working in one sentence.")
print("✅ API Connection Successful!")
print(test)

✅ API Connection Successful!
I am connected and ready to assist you!


In [12]:
# ============================================================
# Template 1 (T1): Basic Advice
# ============================================================

"""
Template Name: Basic Advice
Template ID: T1

Intended Use Case:
    Provide a short, plain-language explanation of the ML model's
    prediction. Designed for general users who need simple,
    actionable advice without medical/technical jargon.

Design Rationale:
    This is the baseline template. It asks GPT to respond in plain
    English only, with no structured sections. The goal is to test
    if minimal prompting still produces useful output. This forms the
    comparison baseline for all other templates.

Prompt Structure:
    A patient has the following characteristics:
    {features}

    Our system predicted: {prediction}

    Please give a simple, plain-language explanation of this result
    and basic advice. Keep the response under 120 words.
    Avoid technical jargon.

Assumptions & Limitations:
    - No system role defined — model behaves as general assistant
    - Does not ask for structured output, so format may vary
    - May be too brief for complex cases
    - Best suited as a quick summary for non-expert users
"""

TEMPLATE_1 = """A patient has the following characteristics:
{features}

Our system predicted: {prediction}

Please give a simple, plain-language explanation of this result and basic advice. Keep the response under 120 words. Avoid technical jargon."""

def run_template_1(features: str, prediction: str) -> str:
    prompt = TEMPLATE_1.format(features=features, prediction=prediction)
    return call_gpt(prompt)  # No system message — intentionally minimal

# ---- Example Input/Output ----
example_features = "Age: 45, Symptoms: chest pain and shortness of breath, BMI: 28.5"
example_prediction = "Cardiology specialist referral recommended"

print("=" * 50)
print("TEMPLATE 1 — Basic Advice")
print("=" * 50)
output_t1_example1 = run_template_1(example_features, example_prediction)
print(output_t1_example1)

TEMPLATE 1 — Basic Advice
Based on your age, symptoms like chest pain and shortness of breath, and your body weight, our system suggests that you should see a heart doctor (a cardiologist). This is important because these symptoms could be serious and need further evaluation. 

In the meantime, try to avoid any strenuous activities, pay attention to how you're feeling, and don’t hesitate to seek immediate help if your symptoms worsen. It’s always best to be safe when it comes to your heart health.


In [13]:
# ============================================================
# Template 2 (T2): Detailed Clinical Explanation
# ============================================================

"""
Template Name: Detailed Clinical Explanation
Template ID: T2

Intended Use Case:
    Provides a structured, thorough explanation for caregivers,
    nursing students, or informed users who want to understand the
    clinical reasoning. Significantly more detailed than T1.

Design Rationale:
    Unlike T1, this template:
    1. Assigns GPT a specific expert role via a system message
    2. Asks for a clearly structured response (4 numbered sections)
    3. Targets an audience with basic health literacy

    The system message primes GPT to behave as a medical advisor,
    improving accuracy and tone consistency across test cases.
    The structured output also makes it easier to evaluate
    completeness in the comparative analysis.

Prompt Structure:
    System: You are a clinical decision-support assistant.
            Provide structured, evidence-based explanations.

    User:
    Patient Profile:
    {features}

    ML Model Prediction: {prediction}

    Provide a structured response with these four sections:
    1. Prediction Explanation — Why did the model predict this?
    2. Key Risk Factors — Which features are most significant?
    3. Recommended Actions — What steps should the patient take?
    4. Warning Signs — What symptoms need immediate attention?

Assumptions & Limitations:
    - System message may cause responses to be overly formal
    - Structured format works best for multi-feature inputs
    - Section 2 (risk factors) relies on model inference, not
      actual feature importance values from the ML model
"""

SYSTEM_T2 = "You are a clinical decision-support assistant. Provide structured, evidence-based explanations to help caregivers and healthcare students understand ML-driven medical predictions."

TEMPLATE_2 = """Patient Profile:
{features}

ML Model Prediction: {prediction}

Provide a structured response with exactly these four sections:
1. Prediction Explanation — Why did the model likely predict this?
2. Key Risk Factors — Which features are most clinically significant?
3. Recommended Actions — What specific steps should be taken?
4. Warning Signs — What symptoms require immediate medical attention?

Keep each section to 2-3 sentences. Target audience: healthcare students and caregivers."""

def run_template_2(features: str, prediction: str) -> str:
    prompt = TEMPLATE_2.format(features=features, prediction=prediction)
    return call_gpt(prompt, system_message=SYSTEM_T2)

# ---- Example Input/Output ----
print("=" * 50)
print("TEMPLATE 2 — Detailed Clinical Explanation")
print("=" * 50)
output_t2_example1 = run_template_2(example_features, example_prediction)
print(output_t2_example1)

TEMPLATE 2 — Detailed Clinical Explanation
### Prediction Explanation
The model likely predicted a referral to a cardiology specialist due to the patient's presenting symptoms of chest pain and shortness of breath, which are commonly associated with cardiovascular conditions. Additionally, the patient's age and elevated BMI may contribute to an increased risk of heart disease, prompting the need for specialized evaluation.

### Key Risk Factors
The most clinically significant features in this case are the patient's age (45 years), symptoms of chest pain and shortness of breath, and a BMI of 28.5, which classifies them as overweight. These factors collectively heighten the likelihood of underlying cardiovascular issues, warranting further investigation by a specialist.

### Recommended Actions
The caregiver should facilitate an urgent referral to a cardiology specialist for comprehensive evaluation, including possible diagnostic tests such as an echocardiogram, stress test, or cardiac c

In [15]:
import json, os

os.makedirs("Generative_AI/prompts", exist_ok=True)
os.makedirs("Generative_AI/example_outputs", exist_ok=True)

# Save templates as JSON
templates = {
    "T1": {
        "name": "Basic Advice",
        "model": "gpt-4o-mini",
        "system_message": None,
        "template": TEMPLATE_1
    },
    "T2": {
        "name": "Detailed Clinical Explanation",
        "model": "gpt-4o-mini",
        "system_message": SYSTEM_T2,
        "template": TEMPLATE_2
    }
}

with open("Generative_AI/prompts/templates.json", "w") as f:
    json.dump(templates, f, indent=2)
print("✅ Templates saved to Generative_AI/prompts/templates.json")

# Save example outputs
with open("Generative_AI/example_outputs/T1_example1.txt", "w") as f:
    f.write(f"Template: T1 — Basic Advice\n")
    f.write(f"Input Features: {example_features}\n")
    f.write(f"Prediction: {example_prediction}\n")
    f.write(f"\n--- OUTPUT ---\n{output_t1_example1}")

with open("Generative_AI/example_outputs/T2_example1.txt", "w") as f:
    f.write(f"Template: T2 — Detailed Clinical Explanation\n")
    f.write(f"Input Features: {example_features}\n")
    f.write(f"Prediction: {example_prediction}\n")
    f.write(f"\n--- OUTPUT ---\n{output_t2_example1}")

print("✅ Example outputs saved to Generative_AI/example_outputs/")

✅ Templates saved to Generative_AI/prompts/templates.json
✅ Example outputs saved to Generative_AI/example_outputs/
